# Step 12 — Pipeline debug: gsplat → SAM3D voxels + mesh + GT alignment

End-to-end visibility of the actual pipeline, using pipeline functions throughout:

1. **Pick** a random `Seen/train` Gaussian + locate its GT point cloud (`PointCloud/PC_<id>.ply`).
2. **Render** a gsplat view at the SAM3D reference camera and display it.
3. **Run SAM3D** (or load from cache) via `ensure_sam3d_reconstruction_for_splat`.
4. **Show voxels** — `slat_coords` scatter coloured by feature magnitude.
5. **Show mesh** — SAM3D `mesh.glb` rendered with pyrender.
6. **Align** — normalise GT to the splat frame, overlay with SAM3D mesh, then run a
   quick ICP to show the residual misalignment and correction.

**Requirements:** CUDA + gsplat + SAM3D weights (`sam3d-pipeline` Docker).
Run **inside** `docker compose run --rm sam3d-pipeline bash`.

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for ROOT in [_cwd, *_cwd.parents]:
    if (ROOT / "pyproject.toml").is_file() and (ROOT / "src").is_dir():
        if str(ROOT / "src") not in sys.path:
            sys.path.insert(0, str(ROOT / "src"))
        break
else:
    raise RuntimeError("Run from the repo or notebooks/.")

from utils.config import load_config
cfg = load_config()
print("ROOT:", ROOT)

## 1 — Pick a random splat + GT point cloud

In [ ]:
import re

from datasets.affordsplat_local_dataset import resolve_affordsplat_root, sample_random_affordsplat_row

RANDOM_SEED = None   # int for reproducible draw, None for fresh each run
SUBSET = "Seen"
SPLIT  = "train"

root = resolve_affordsplat_root(cfg)
if root is None:
    raise FileNotFoundError(
        "AffordSplat root not found. Set AFFORDANCE_AFFORDSPLAT_ROOT or paths.affordsplat_root."
    )

SPLAT_PLY = None
GT_PLY    = None
ROW       = None

for attempt in range(128):
    seed_try = None if RANDOM_SEED is None else int(RANDOM_SEED) + attempt
    row = sample_random_affordsplat_row(affordsplat_root=root, seed=seed_try,
                                        subset=SUBSET, split=SPLIT, cfg=cfg)
    if row is None:
        break
    nid = row.extras["affordsplat_gaussian_numeric_id"]
    gt  = row.splat_path.parent.parent / "PointCloud" / f"PC_{nid}.ply"
    if row.splat_path.is_file() and gt.is_file():
        SPLAT_PLY, GT_PLY, ROW = row.splat_path, gt, row
        break

if SPLAT_PLY is None:
    raise FileNotFoundError("No splat+GT pair found after 128 draws. Check AffordSplat mount.")

print("sample_id :", ROW.sample_id)
print("verb      :", ROW.verb)
print("splat     :", SPLAT_PLY)
print("GT        :", GT_PLY)
if ROW.affordance_gs_anno_path and ROW.affordance_gs_anno_path.is_file():
    print("aff anno  :", ROW.affordance_gs_anno_path)

## 2 — Render the SAM3D reference view with gsplat

In [ ]:
import gc
import matplotlib.pyplot as plt
import numpy as np
import torch
from dataclasses import replace

from rendering.gaussian_gsplat_renderer import render_gaussian_splat_gsplat_views
from rendering.renderer import build_render_config

REFERENCE_VIEW = 0  # only view rendered

mrc = replace(
    build_render_config(cfg),
    num_views=1,
    orbit_azimuth_offsets_deg=(45.0,),
    orbit_axis_mode="world",
    orbit_axis=None,
    orbit_ring_rotation_deg=90.0,
    orbit_ring_rotation_axis=(1.0, 0.0, 0.0),
)

gsplat_views = render_gaussian_splat_gsplat_views(
    SPLAT_PLY, mrc, max_points=500_000, normalize_scene=True, seed=0
)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(gsplat_views[0].rgb)
ax.set_title(f"gsplat +45° (SAM3D input)  —  {ROW.sample_id}", fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.show()

REFERENCE_RGB = np.array(gsplat_views[REFERENCE_VIEW].rgb)
SCENE_CENTER  = gsplat_views[REFERENCE_VIEW].meta.get("scene_normalize_center") if hasattr(gsplat_views[REFERENCE_VIEW], "meta") else None
SCENE_SCALE   = gsplat_views[REFERENCE_VIEW].meta.get("scene_normalize_scale")  if hasattr(gsplat_views[REFERENCE_VIEW], "meta") else None
print(f"reference view shape: {REFERENCE_RGB.shape}")

# Free GPU memory before SAM3D loads its ~40 GB model
del gsplat_views
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free after render: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

## 3 — Run SAM3D (generate-or-load)

In [ ]:
from reconstruction.gsplat_sam3d_batch import ensure_sam3d_reconstruction_for_splat

SAM3D_OUTPUT_ROOT = ROOT / "exports" / "gsplat_sam3d_runs"

result = ensure_sam3d_reconstruction_for_splat(
    SPLAT_PLY,
    output_root=SAM3D_OUTPUT_ROOT,
    cfg=cfg,
    force=False,
    reference_view_index=REFERENCE_VIEW,
    max_points=500_000,
    sam3d_seed=42,
)

RECON_DIR = result["reconstruction_dir"]
RUN_DIR   = result["run_dir"]
print("status      :", result["status"])
print("recon dir   :", RECON_DIR)
print("mesh        :", result["mesh_glb"])

## 4 — SAM3D voxels: `slat_coords` scatter coloured by feature magnitude

In [ ]:
import torch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from reconstruction.mesh_utils import load_latents

latents = load_latents(RECON_DIR)
slat_feats  = latents["slat_feats"].float()   # (N, 8)
slat_coords = latents["slat_coords"].float()   # (N, 3)
slat_vf     = latents.get("slat_vertex_features")  # (V, 8) or None

print(f"slat_feats  : {tuple(slat_feats.shape)}   (N occupied voxels × 8)")
print(f"slat_coords : {tuple(slat_coords.shape)}")
if slat_vf is not None:
    print(f"slat_vertex_features: {tuple(slat_vf.shape)}  (V mesh vertices × 8)")

coords = slat_coords.numpy()
magnitudes = slat_feats.norm(dim=-1).numpy()  # (N,)
mag_norm = (magnitudes - magnitudes.min()) / (magnitudes.ptp() + 1e-8)

fig = plt.figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection="3d")
sc  = ax.scatter(
    coords[:, 0], coords[:, 1], coords[:, 2],
    c=mag_norm, cmap="plasma", s=18, alpha=0.85, depthshade=True,
)
plt.colorbar(sc, ax=ax, shrink=0.6, label="feature L2 (normalised)")
ax.set_title(f"SAM3D SLAT voxels — {len(coords)} occupied  ({ROW.sample_id})")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
try:
    ax.set_box_aspect((1, 1, 1))
except Exception:
    pass
ax.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

## 5 — SAM3D mesh (pyrender)

In [ ]:
import trimesh
from PIL import Image as PILImage

from datasets.mesh_loading import mesh_data_from_trimesh
from reconstruction.mesh_utils import reconstruction_paths
from rendering.mesh_renderer import MeshRenderConfig, MeshRenderer

mesh_glb = reconstruction_paths(RECON_DIR)["mesh"]

def _as_trimesh(loaded):
    if isinstance(loaded, trimesh.Scene):
        parts = [g for g in loaded.geometry.values()
                 if isinstance(g, trimesh.Trimesh) and len(g.vertices)]
        if not parts:
            raise ValueError("empty Scene")
        return trimesh.util.concatenate(parts) if len(parts) > 1 else parts[0]
    return loaded

tm = _as_trimesh(trimesh.load(str(mesh_glb), process=False))
print(f"mesh.glb: {len(tm.vertices)} vertices, {len(tm.faces)} faces")

preview_mrc = MeshRenderConfig(
    image_size=640, fov_deg=55.0, num_views=1,
    camera_radius=2.2, elevation_deg=35.0,
    orbit_axis_mode="world", orbit_axis=None,
    orbit_ring_rotation_deg=90.0,
    orbit_ring_rotation_axis=(1.0, 0.0, 0.0),
    render_geometry_aux=False,
)

try:
    mesh_data = mesh_data_from_trimesh(tm)
    views = MeshRenderer(preview_mrc).render(mesh_data)
    rgb = np.asarray(views[0].rgb)[:, :, :3]
    depth = np.asarray(views[0].depth, dtype=np.float32)
    bg = np.array([240, 240, 244], dtype=np.uint8)
    rgb = np.where((depth > 0)[..., None], rgb, bg).astype(np.uint8)

    preview_png = RECON_DIR / "preview_sam3d_mesh.png"
    PILImage.fromarray(rgb).save(preview_png)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(REFERENCE_RGB)
    axes[0].set_title("gsplat reference (SAM3D input)", fontsize=10)
    axes[0].axis("off")
    axes[1].imshow(rgb)
    axes[1].set_title("SAM3D mesh (pyrender)", fontsize=10)
    axes[1].axis("off")
    plt.suptitle(ROW.sample_id, fontsize=11)
    plt.tight_layout()
    plt.show()
    print("Wrote:", preview_png)
except Exception as exc:
    print(f"pyrender failed ({type(exc).__name__}: {exc}).")
    print("Open", mesh_glb, "in an external viewer, or run with EGL/OSMesa.")

MESH_TRIMESH = tm

## 6 — Align SAM3D mesh with GT point cloud

Both representations are brought to the **splat-normalised frame**
(splat means centred and scaled to unit radius).  
The SAM3D mesh is already exported in that frame (inverse ring rotation applied in `save_reconstruction`).  
The GT point cloud is normalised with the same centre/scale derived from splat means.

We then run a lightweight ICP (SVD rigid alignment, `scipy` nearest-neighbours) to quantify
and correct residual misalignment.

In [ ]:
from rendering.gaussian_ply import load_gaussian_splat_ply
from rendering.gsplat_viewpoint_selection import (
    _scene_normalize_params,
    apply_scene_normalize,
    load_gt_point_cloud,
)

# ---- GT point cloud in splat-normalised frame ----
gt_raw  = load_gt_point_cloud(GT_PLY)                         # (M, 3) raw
means   = load_gaussian_splat_ply(SPLAT_PLY).means           # splat Gaussian centres
sn_c, sn_s = _scene_normalize_params(means, target_radius=0.95)
gt_norm = apply_scene_normalize(gt_raw, sn_c, sn_s)           # (M, 3) in splat frame

# ---- SAM3D mesh in splat frame (already there after export) ----
mesh_verts = np.asarray(MESH_TRIMESH.vertices, dtype=np.float64)

print(f"GT points (normalised) : {gt_norm.shape}")
print(f"mesh vertices          : {mesh_verts.shape}")
print(f"GT centre  : {gt_norm.mean(axis=0).round(3)}")
print(f"mesh centre: {mesh_verts.mean(axis=0).round(3)}")

In [ ]:
from mpl_toolkits.mplot3d import art3d

def _add_mesh_faces(ax, tm, *, max_faces=8000, seed=0, **kw):
    faces = np.asarray(tm.faces)
    if len(faces) > max_faces:
        rng = np.random.default_rng(seed)
        faces = faces[rng.choice(len(faces), max_faces, replace=False)]
    coll = art3d.Poly3DCollection(np.asarray(tm.vertices)[faces], **kw)
    ax.add_collection3d(coll)

def _axis_equal_3d(ax, pts_list, pad=1.15):
    all_pts = np.concatenate(pts_list, axis=0)
    c = 0.5 * (all_pts.min(0) + all_pts.max(0))
    r = max(float((all_pts.max(0) - all_pts.min(0)).max()) * 0.5 * pad, 1e-4)
    ax.set_xlim(c[0]-r, c[0]+r)
    ax.set_ylim(c[1]-r, c[1]+r)
    ax.set_zlim(c[2]-r, c[2]+r)
    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass

fig, axes = plt.subplots(1, 2, figsize=(12, 5), subplot_kw={"projection": "3d"})

for ax, title in zip(axes, ["GT point cloud (normalised)", "SAM3D mesh (normalised)"]):
    ax.set_title(title, fontsize=10)
    ax.view_init(elev=22, azim=-60)

# left: GT only
subsample = np.random.default_rng(0).choice(len(gt_norm), min(4000, len(gt_norm)), replace=False)
axes[0].scatter(gt_norm[subsample, 0], gt_norm[subsample, 1], gt_norm[subsample, 2],
                s=3, c="darkorange", alpha=0.7, depthshade=False)
_axis_equal_3d(axes[0], [gt_norm])

# right: mesh only
_add_mesh_faces(axes[1], MESH_TRIMESH, facecolor="steelblue",
                edgecolor="0.3", linewidths=0.04, alpha=0.9)
_axis_equal_3d(axes[1], [mesh_verts])

plt.suptitle(f"Before alignment — {ROW.sample_id}", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.spatial import cKDTree

def icp_rigid(
    src: np.ndarray,   # (N, 3) points to align
    tgt: np.ndarray,   # (M, 3) target point cloud
    *,
    n_iter: int = 60,
    n_samples: int = 2000,
    rng_seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Rigid ICP (rotation + translation only, no scale).  
    Returns (aligned_src, 4x4 cumulative transform)."""
    rng   = np.random.default_rng(rng_seed)
    tree  = cKDTree(tgt)
    pts   = src.copy().astype(np.float64)
    T_cum = np.eye(4)

    for _ in range(n_iter):
        idx   = rng.choice(len(pts), min(n_samples, len(pts)), replace=False)
        query = pts[idx]
        _, nn = tree.query(query, k=1)
        tgt_q = tgt[nn]

        src_c  = query.mean(0);  tgt_c = tgt_q.mean(0)
        H      = (query - src_c).T @ (tgt_q - tgt_c)
        U, _, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:          # reflection fix
            Vt[-1] *= -1
            R = Vt.T @ U.T
        t = tgt_c - R @ src_c

        pts = (R @ pts.T).T + t
        T_step = np.eye(4);  T_step[:3, :3] = R;  T_step[:3, 3] = t
        T_cum  = T_step @ T_cum

    return pts, T_cum


# Centre both before ICP (removes gross translation)
gt_c   = gt_norm.mean(0)
mesh_c = mesh_verts.mean(0)
mesh_verts_shifted = mesh_verts - mesh_c + gt_c   # rough centroid alignment as ICP warm-start

mesh_aligned, T_icp = icp_rigid(mesh_verts_shifted, gt_norm)

# Chamfer-like mean nearest-neighbour distance before and after
tree_gt = cKDTree(gt_norm)
d_before, _ = tree_gt.query(mesh_verts_shifted, k=1)
d_after,  _ = tree_gt.query(mesh_aligned,       k=1)
print(f"Mean NN distance — before ICP: {d_before.mean():.4f}   after: {d_after.mean():.4f}")

In [ ]:
# Build a temporary trimesh with ICP-aligned vertices for plotting
tm_aligned = MESH_TRIMESH.copy()
tm_aligned.vertices = mesh_aligned

fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"projection": "3d"})
titles = ["Before ICP (centroid only)", "After ICP (rigid align)"]
mesh_versions = [mesh_verts_shifted, mesh_aligned]
mesh_tms = [MESH_TRIMESH, tm_aligned]

for ax, title, mv, mt in zip(axes, titles, mesh_versions, mesh_tms):
    # GT cloud
    ax.scatter(gt_norm[subsample, 0], gt_norm[subsample, 1], gt_norm[subsample, 2],
               s=3, c="darkorange", alpha=0.6, depthshade=False, label="GT")
    # mesh (use the version with updated vertices)
    _tm_plot = mt.copy()
    _tm_plot.vertices = mv
    _add_mesh_faces(ax, _tm_plot, facecolor="steelblue",
                    edgecolor="0.3", linewidths=0.04, alpha=0.45)
    _axis_equal_3d(ax, [gt_norm, mv])
    ax.set_title(title, fontsize=10)
    ax.view_init(elev=22, azim=-60)
    ax.legend(fontsize=8, loc="upper right")

plt.suptitle(
    f"SAM3D mesh (blue) + GT point cloud (orange) — {ROW.sample_id}\n"
    f"NN dist  before={d_before.mean():.4f}  after={d_after.mean():.4f}",
    fontsize=10,
)
plt.tight_layout()
plt.show()